In [1]:
import pandas as pd
import numpy as np
import os

BASE_PATH = "../dataset"

def get_path(folder, file):
    return os.path.join(BASE_PATH, folder, file)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
def load_all_datasets():
    print("Đang tải dữ liệu...")
    products = pd.read_csv(get_path('raw', 'products.csv'))
    customers = pd.read_csv(get_path('raw', 'customers.csv'), parse_dates=['signup_date'])
    geography = pd.read_csv(get_path('raw', 'geography.csv'))
    promotion = pd.read_csv(get_path('raw', 'promotions.csv'))
    
    orders = pd.read_csv(get_path('raw', 'orders.csv'), parse_dates=['order_date'])
    order_items = pd.read_csv(get_path('raw', 'order_items.csv'))
    returns = pd.read_csv(get_path('raw', 'returns.csv'), parse_dates=['return_date'])
    payments = pd.read_csv(get_path('raw', 'payments.csv'))
    reviews = pd.read_csv(get_path('raw', 'reviews.csv'))
    shipments = pd.read_csv(get_path('raw', 'shipments.csv'))
    
    web_traffic = pd.read_csv(get_path('raw', 'web_traffic.csv'), parse_dates=['date'])
    inventory = pd.read_csv(get_path('raw', 'inventory.csv'))

    sales_train = pd.read_csv(get_path('raw', 'sales.csv'), parse_dates=['Date'])
    
    print("Tải dữ liệu thành công!")
    return products, customers, geography, promotion, orders, order_items, returns, payments, reviews, shipments, web_traffic, inventory, sales_train

products, customers, geography, promotion, orders, order_items, returns, payments, reviews, shipments, web_traffic, inventory, sales_train = load_all_datasets()

Đang tải dữ liệu...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13244\891705054.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(get_path('raw', 'order_items.csv'))


Tải dữ liệu thành công!


In [ ]:
def solve_mcqs():

    # Q1: Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap)
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    orders_sorted = orders.sort_values(['customer_id', 'order_date'])
    orders_sorted['prev_order_date'] = orders_sorted.groupby('customer_id')['order_date'].shift(1)
    orders_sorted['inter_order_gap'] = (orders_sorted['order_date'] - orders_sorted['prev_order_date']).dt.days
    q1_ans = orders_sorted['inter_order_gap'].median()
    print(f"Q1: Trung vị khoảng cách giữa 2 lần mua: {q1_ans} ngày")

    # Q2: Segment có margin cao nhất (price - cogs)/price
    products['margin'] = (products['price'] - products['cogs']) / products['price']
    q2_ans = products.groupby('segment')['margin'].mean().idxmax()
    print(f"Q2: Segment có lợi nhuận cao nhất: {q2_ans}")

    # Q3: Lý do trả hàng Streetwear (Top lý do)
    sw_prods = products[products['category'] == 'Streetwear']['product_id']
    sw_returns = returns[returns['product_id'].isin(sw_prods)]
    q3_ans = sw_returns['return_reason'].value_counts().idxmax()
    print(f"Q3: Lý do trả hàng phổ biến nhất: {q3_ans}")

    # Q4: Traffic source có bounce_rate thấp nhất
    q4_ans = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
    print(f"Q4: Nguồn traffic chất lượng nhất: {q4_ans}")

    # Q5: % đơn hàng có khuyến mãi (promo_id is not null)
    q5_ans = (order_items['promo_id'].notna().sum() / len(order_items)) * 100
    print(f"Q5: Tỷ lệ áp dụng khuyến mãi: {q5_ans:.1f}%")

    # Q6: Nhóm tuổi có số đơn hàng TB cao nhất
    order_counts = orders.groupby('customer_id').size().reset_index(name='n_orders')
    age_data = customers.merge(order_counts, on='customer_id', how='left').fillna(0)
    q6_ans = age_data[age_data['age_group'].notna()].groupby('age_group')['n_orders'].mean().idxmax()
    print(f"Q6: Nhóm tuổi mua nhiều nhất: {q6_ans}")

    # Q7: Vùng (Region) có doanh thu cao nhất
    df_sales = order_items.merge(orders[['order_id', 'zip']], on='order_id')
    df_sales = df_sales.merge(geography[['zip', 'region']], on='zip')
    df_sales['net_rev'] = df_sales['quantity'] * df_sales['unit_price']
    q7_ans = df_sales.groupby('region')['net_rev'].sum().idxmax()
    print(f"Q7: Vùng đóng góp doanh thu lớn nhất: {q7_ans}")

    # Q8: Payment method của đơn 'cancelled'
    q8_ans = orders[orders['order_status'] == 'cancelled']['payment_method'].value_counts().idxmax()
    print(f"Q8: Phương thức thanh toán hay bị hủy: {q8_ans}")

    # Q9: Size có tỷ lệ trả hàng cao nhất (Returns / Order_items)
    ret_size = returns.merge(products[['product_id', 'size']], on='product_id')['size'].value_counts()
    sold_size = order_items.merge(products[['product_id', 'size']], on='product_id')['size'].value_counts()
    q9_ans = (ret_size / sold_size).idxmax()
    print(f"Q9: Kích cỡ có tỷ lệ trả hàng cao nhất: {q9_ans}")

    # Q10: Số kỳ trả góp có giá trị thanh toán trung bình cao nhất
    q10_ans = payments.groupby('installments')['payment_value'].mean().idxmax()
    print(f"Q10: Kế hoạch trả góp giá trị nhất: {q10_ans} kỳ")

solve_mcqs()

Q1: Trung vị khoảng cách giữa 2 lần mua: 144.0 ngày
Q2: Segment có lợi nhuận cao nhất: Standard
Q3: Lý do trả hàng phổ biến nhất: wrong_size
Q4: Nguồn traffic chất lượng nhất: email_campaign
Q5: Tỷ lệ áp dụng khuyến mãi: 38.7%
Q6: Nhóm tuổi mua nhiều nhất: 55+
Q7: Vùng đóng góp doanh thu lớn nhất: East
Q8: Phương thức thanh toán hay bị hủy: credit_card
Q9: Kích cỡ có tỷ lệ trả hàng cao nhất: S
Q10: Kế hoạch trả góp giá trị nhất: 6 kỳ
